In [29]:
import pandas as pd
import numpy as np
import re

#### LOAD AND INSPECT DATA

In [30]:

# Kaggle multi-resistance dataset
kaggle_df = pd.read_csv('Data/Bacteria_dataset_Multiresictance.csv')
print(kaggle_df.shape)
print(kaggle_df.dtypes)
print(kaggle_df.isnull().sum())

# Mendeley dataset (xlsx)
mendeley_df = pd.read_excel('Data/Dataset.xlsx')
print(mendeley_df.head())

# CARD ontology TSV 
card_df = pd.read_csv('Data/aro_index.tsv', sep='\t')
# Useful columns: ARO Name, Drug Class, Resistance Mechanism, AMR Gene Family
print(card_df[['ARO Name', 'Drug Class', 'Resistance Mechanism', 'AMR Gene Family']].head(10))

(10710, 27)
ID                   object
Name                 object
Email                object
Address              object
age/gender           object
Souches              object
Diabetes             object
Hypertension         object
Hospital_before      object
Infection_Freq       object
AMX/AMP              object
AMC                  object
CZ                   object
FOX                  object
CTX/CRO              object
IPM                  object
GEN                  object
AN                   object
Acide nalidixique    object
ofx                  object
CIP                  object
C                    object
Co-trimoxazole       object
Furanes              object
colistine            object
Collection_Date      object
Notes                object
dtype: object
ID                     0
Name                   0
Email                  0
Address                0
age/gender           621
Souches              640
Diabetes             635
Hypertension         630
Hospital_before   

In [31]:
print(kaggle_df['Souches'].value_counts(dropna=False).head(30))

Souches
NaN                              640
?                                 63
missing                           60
S1836 Klebsiella pneumoniae        3
S4809 Escherichia coli             3
S6976 Escherichia coli             3
S687 E. coli                       3
S8071 Escherichia coli             3
S3413 Escherichia coli             3
S564 Escherichia coli              3
S4777 Escherichia coli             3
S6533 Proteus mirabilis            3
S6401 Escherichia coli             3
S4989 Acinetobacter baumannii      3
S7936 Escherichia coli             3
S6855 Escherichia coli             3
S3965 Escherichia coli             3
S597 Escherichia coli              3
S10015 Pseudomonas aeruginosa      3
S2424 Prot.eus mirabilis           3
S7821 E. coli                      3
S4518 Escherichia coli             3
S2180 Escherichia coli             3
S10079 Escherichia coli            3
S7731 Escherichia coli             3
S980 Escherichia coli              2
S3858 Proteus mirabilis       

### Clean and Structure Kaggle dataset on Clinical Multi Antibiotic resistance

#### Standardize species/antibiotic names

In [32]:
KAGGLE_AB_RENAME = {
    'AMX/AMP'          : 'amoxicillin_ampicillin',
    'AMC'              : 'amoxicillin_clavulanate',
    'CZ'               : 'cefazolin',
    'FOX'              : 'cefoxitin',
    'CTX/CRO'          : 'cefotaxime_ceftriaxone',
    'IPM'              : 'imipenem',
    'GEN'              : 'gentamicin',
    'AN'               : 'amikacin',
    'Acide nalidixique': 'nalidixic_acid',
    'ofx'              : 'ofloxacin',
    'CIP'              : 'ciprofloxacin',
    'C'                : 'chloramphenicol',
    'Co-trimoxazole'   : 'cotrimoxazole',
    'Furanes'          : 'nitrofurantoin',
    'colistine'        : 'colistin',
}
 
KAGGLE_META_RENAME = {
    'ID'             : 'patient_id',
    'Name'           : 'patient_name',
    'Email'          : 'patient_email',
    'Address'        : 'patient_address',
    'age/gender'     : 'age_gender',
    'Souches'        : 'bacteria_strain_raw',
    'Diabetes'       : 'has_diabetes',
    'Hypertension'   : 'has_hypertension',
    'Hospital_before': 'prior_hospitalisation',
    'Infection_Freq' : 'infection_frequency',
    'Collection_Date': 'collection_date',
    'Notes'          : 'notes',
}
 
kaggle_df.rename(columns={**KAGGLE_AB_RENAME, **KAGGLE_META_RENAME}, inplace=True)
 
KAGGLE_AB_COLS = list(KAGGLE_AB_RENAME.values())   # 15 antibiotic columns
print(f"Antibiotic columns : {KAGGLE_AB_COLS}")


Antibiotic columns : ['amoxicillin_ampicillin', 'amoxicillin_clavulanate', 'cefazolin', 'cefoxitin', 'cefotaxime_ceftriaxone', 'imipenem', 'gentamicin', 'amikacin', 'nalidixic_acid', 'ofloxacin', 'ciprofloxacin', 'chloramphenicol', 'cotrimoxazole', 'nitrofurantoin', 'colistin']


#### CLEAN R/I/S VALUES


In [33]:
RIS_MAP = {
    # Resistant
    'r': 2, 'resistant': 2,
    # Intermediate
    'i': 1, 'intermediate': 1,
    # Susceptible
    's': 0, 'susceptible': 0,
}
 
SENTINEL = -1   # "not tested" — never impute resistance labels
 
def encode_ris(val):
    """
    Normalise any R/I/S variant to 2/1/0.
    Unknown / missing → SENTINEL (-1).
    """
    if pd.isna(val):
        return SENTINEL
    cleaned = str(val).strip().lower()
    if cleaned in ('', '?', 'missing', 'nan', 'none', 'nd', '-'):
        return SENTINEL
    return RIS_MAP.get(cleaned, SENTINEL)
 
for col in KAGGLE_AB_COLS:
    before_null = kaggle_df[col].isna().sum()
    kaggle_df[col] = kaggle_df[col].apply(encode_ris)
    sentinel_count = (kaggle_df[col] == SENTINEL).sum()
    print(f"  {col:<35} nulls_before={before_null:>3}  sentinel_after={sentinel_count:>3}")
 
# Sanity check — no unexpected values
for col in KAGGLE_AB_COLS:
    bad = set(kaggle_df[col].unique()) - {0, 1, 2, SENTINEL}
    if bad:
        print(f"  WARNING unexpected values in {col}: {bad}")
 

  amoxicillin_ampicillin              nulls_before=658  sentinel_after=753
  amoxicillin_clavulanate             nulls_before=632  sentinel_after=753
  cefazolin                           nulls_before=628  sentinel_after=753
  cefoxitin                           nulls_before=625  sentinel_after=753
  cefotaxime_ceftriaxone              nulls_before=624  sentinel_after=753
  imipenem                            nulls_before=633  sentinel_after=753
  gentamicin                          nulls_before=643  sentinel_after=753
  amikacin                            nulls_before=628  sentinel_after=753
  nalidixic_acid                      nulls_before=622  sentinel_after=753
  ofloxacin                           nulls_before=618  sentinel_after=753
  ciprofloxacin                       nulls_before=633  sentinel_after=753
  chloramphenicol                     nulls_before=629  sentinel_after=753
  cotrimoxazole                       nulls_before=640  sentinel_after=753
  nitrofurantoin         

#### CLEAN KAGGLE CLINICAL META DATA

In [34]:
# ── age / gender ──────────────────────────────────────────
def parse_age(val):
    if pd.isna(val): return np.nan
    digits = re.findall(r'\d+', str(val))
    return int(digits[0]) if digits else np.nan
 
def parse_gender(val):
    if pd.isna(val): return np.nan
    v = str(val).upper()
    if 'F' in v: return 'F'
    if 'M' in v: return 'M'
    return np.nan
 
kaggle_df['age']    = kaggle_df['age_gender'].apply(parse_age)
kaggle_df['gender'] = kaggle_df['age_gender'].apply(parse_gender)
 
# Clip implausible ages (< 0 or > 120)
kaggle_df.loc[~kaggle_df['age'].between(0, 120, inclusive='both'), 'age'] = np.nan
 
# ── binary clinical flags ─────────────────────────────────
def to_binary(val):
    if pd.isna(val): return np.nan
    v = str(val).strip().lower()
    if v in ('yes', '1', 'true', 'oui', 'y'):  return 1
    if v in ('no',  '0', 'false','non', 'n'):   return 0
    return np.nan
 
for col in ['has_diabetes', 'has_hypertension', 'prior_hospitalisation']:
    kaggle_df[col] = kaggle_df[col].apply(to_binary)
 
# ── infection frequency → ordinal ────────────────────────
FREQ_MAP = {
    'never': 0, 'jamais': 0, 'no': 0,
    'sometimes': 1, 'parfois': 1, 'occasional': 1, 'occasionnel': 1,
    'often': 2, 'souvent': 2, 'frequent': 2, 'always': 2,
}
kaggle_df['infection_frequency'] = (
    kaggle_df['infection_frequency']
    .astype(str).str.strip().str.lower()
    .map(FREQ_MAP)
)
 
print("  age        — parsed from age_gender")
print("  gender     — parsed from age_gender")
print("  binary flags (diabetes, hypertension, hospitalisation) — encoded")
print("  infection_frequency — ordinal encoded")

  age        — parsed from age_gender
  gender     — parsed from age_gender
  binary flags (diabetes, hypertension, hospitalisation) — encoded
  infection_frequency — ordinal encoded


#### CLEAN bacteria_strain (Souches)

In [35]:
SPECIES_SYNONYMS = {
    'e. coli'              : 'escherichia coli',
    'e.coli'               : 'escherichia coli',
    'ecoli'                : 'escherichia coli',
    'escherichia coli'     : 'escherichia coli',
    'prot.eus mirabilis'   : 'proteus mirabilis',   # typo in your data
    'p. mirabilis'         : 'proteus mirabilis',
    'proteus mirabilis'    : 'proteus mirabilis',
    'k. pneumoniae'        : 'klebsiella pneumoniae',
    'klebsiella pneumoniae': 'klebsiella pneumoniae',
    'p. aeruginosa'        : 'pseudomonas aeruginosa',
    'pseudomonas aeruginosa':'pseudomonas aeruginosa',
    'a. baumannii'         : 'acinetobacter baumannii',
    'acinetobacter baumannii':'acinetobacter baumannii',
    'e. faecalis'          : 'enterococcus faecalis',
    'enterococcus faecalis': 'enterococcus faecalis',
    'e. faecium'           : 'enterococcus faecium',
    's. aureus'            : 'staphylococcus aureus',
    'staphylococcus aureus': 'staphylococcus aureus',
    'e. cloacae'           : 'enterobacter cloacae',
    'enterobacter cloacae' : 'enterobacter cloacae',
}
 
def clean_species(val):
    if pd.isna(val): return np.nan
    val = str(val).strip()
    if val in ('?', 'missing', 'nan', '', 'none'): return np.nan
    # Strip leading sample ID prefix: "S1836 " or "S10079 "
    val = re.sub(r'^S\d+\s+', '', val).strip().lower()
    return SPECIES_SYNONYMS.get(val, val)
 
kaggle_df['bacteria_strain'] = kaggle_df['bacteria_strain_raw'].apply(clean_species)
 
print("Species distribution after cleaning:")
print(kaggle_df['bacteria_strain'].value_counts(dropna=False).head(15).to_string())


Species distribution after cleaning:
bacteria_strain
escherichia coli           5255
enterobacteria spp.         802
NaN                         763
proteus mirabilis           644
klebsiella pneumoniae       565
citrobacter spp.            481
e.coi                       432
e.cli                       396
morganella morganii         305
serratia marcescens         256
pseudomonas aeruginosa      200
acinetobacter baumannii     181
enter.bacteria spp.         100
enteobacteria spp.           95
klbsiella pneumoniae         74


#### DROP ROWS WITH NO LAB DATA

In [36]:
n_before = len(kaggle_df)
tested = kaggle_df[KAGGLE_AB_COLS].ne(SENTINEL).sum(axis=1)
kaggle_df = kaggle_df[tested >= 2].reset_index(drop=True)
print(f"  Rows before: {n_before}  →  after (≥2 tested antibiotics): {len(kaggle_df)}")

  Rows before: 10710  →  after (≥2 tested antibiotics): 9957


#### Final Columns on Kaggle clinical data

In [37]:
KAGGLE_KEEP = (
    ['patient_id', 'bacteria_strain', 'age', 'gender',
     'has_diabetes', 'has_hypertension', 'prior_hospitalisation',
     'infection_frequency', 'collection_date']
    + KAGGLE_AB_COLS
)
# Only keep columns that actually exist
KAGGLE_KEEP = [c for c in KAGGLE_KEEP if c in kaggle_df.columns]
kaggle_final = kaggle_df[KAGGLE_KEEP].copy()
kaggle_final['source'] = 'kaggle'

### Clean and Structure Mendeley dataset

#### RENAME + CONVERT mm → R/I/S

In [38]:
MENDELEY_AB_RENAME = {
    'IMIPENEM'     : 'imipenem',
    'CEFTAZIDIME'  : 'ceftazidime',
    'GENTAMICIN'   : 'gentamicin',
    'AUGMENTIN'    : 'amoxicillin_clavulanate',
    'CIPROFLOXACIN': 'ciprofloxacin',
}
mendeley_df.rename(columns=MENDELEY_AB_RENAME, inplace=True)
mendeley_df.rename(columns={'Location': 'sample_location'}, inplace=True)
 
MENDELEY_AB_COLS = list(MENDELEY_AB_RENAME.values())

#### EUCAST v14 breakpoints — Enterobacterales, disk diffusion (mm)

In [39]:
EUCAST_BP = {
    'imipenem'               : {'S': 22, 'R': 19},
    'ceftazidime'            : {'S': 21, 'R': 17},
    'gentamicin'             : {'S': 18, 'R': 14},
    'amoxicillin_clavulanate': {'S': 18, 'R': 14},
    'ciprofloxacin'          : {'S': 25, 'R': 21},
}
 
def mm_to_ris(val, antibiotic):
    if pd.isna(val): return SENTINEL
    try:
        v = float(str(val).strip())
    except ValueError:
        return SENTINEL
    bp = EUCAST_BP[antibiotic]
    if v >= bp['S']:   return 0   # Susceptible
    elif v <= bp['R']: return 2   # Resistant
    else:              return 1   # Intermediate
 
for col in MENDELEY_AB_COLS:
    mendeley_df[col] = mendeley_df[col].apply(lambda x: mm_to_ris(x, col))
    dist = mendeley_df[col].value_counts().to_dict()
    label_map = {-1: 'not_tested', 0: 'S', 1: 'I', 2: 'R'}
    print(f"  {col:<35} { {label_map.get(k,k): v for k,v in sorted(dist.items())} }")
 

  imipenem                            {'S': 180, 'I': 60, 'R': 34}
  ceftazidime                         {'S': 30, 'I': 33, 'R': 211}
  gentamicin                          {'S': 149, 'I': 68, 'R': 57}
  amoxicillin_clavulanate             {'S': 63, 'I': 46, 'R': 165}
  ciprofloxacin                       {'S': 73, 'I': 61, 'R': 140}


#### SELECT FINAL COLUMNS from Mendeley dataset

In [40]:
mendeley_final = mendeley_df[['sample_location'] + MENDELEY_AB_COLS].copy()
mendeley_final['source']          = 'mendeley'
mendeley_final['bacteria_strain'] = np.nan   # no species info in Mendeley
 
# Fill the 10 antibiotics Mendeley doesn't have with SENTINEL
kaggle_only_ab = [c for c in KAGGLE_AB_COLS if c not in MENDELEY_AB_COLS]
for col in kaggle_only_ab:
    mendeley_final[col] = SENTINEL
 
# Also fill Kaggle-only metadata with NaN
for col in ['patient_id','age','gender','has_diabetes','has_hypertension',
            'prior_hospitalisation','infection_frequency','collection_date']:
    mendeley_final[col] = np.nan

### Merge Both Datasets

In [41]:
ALL_AB_COLS = KAGGLE_AB_COLS   # 15 cols — master list
 
# Align both dataframes to same column set
SHARED_COLS = (
    ['source', 'bacteria_strain', 'sample_location', 'patient_id',
     'age', 'gender', 'has_diabetes', 'has_hypertension',
     'prior_hospitalisation', 'infection_frequency', 'collection_date']
    + ALL_AB_COLS
)
 
def align_cols(df, cols):
    for c in cols:
        if c not in df.columns:
            df[c] = np.nan
    return df[cols]
 
kaggle_aligned   = align_cols(kaggle_final,   SHARED_COLS)
mendeley_aligned = align_cols(mendeley_final, SHARED_COLS)
 
merged = pd.concat([kaggle_aligned, mendeley_aligned],
                   ignore_index=True)
 
# Ensure antibiotic cols are integer (safe now — all values are -1/0/1/2)
for col in ALL_AB_COLS:
    merged[col] = pd.to_numeric(merged[col], errors='coerce').fillna(SENTINEL).astype(int)
 
print(f"  Kaggle rows   : {len(kaggle_aligned)}")
print(f"  Mendeley rows : {len(mendeley_aligned)}")
print(f"  Merged total  : {len(merged)}")
print(f"  Columns       : {merged.shape[1]}")

  Kaggle rows   : 9957
  Mendeley rows : 274
  Merged total  : 10231
  Columns       : 26


### ENGINEERED FEATURES 

In [42]:
DRUG_CLASSES = {
    'beta_lactam'   : ['amoxicillin_ampicillin', 'amoxicillin_clavulanate',
                       'cefazolin', 'cefoxitin', 'cefotaxime_ceftriaxone', 'imipenem'],
    'aminoglycoside': ['gentamicin', 'amikacin'],
    'quinolone'     : ['nalidixic_acid', 'ofloxacin', 'ciprofloxacin'],
    'other'         : ['chloramphenicol', 'cotrimoxazole',
                       'nitrofurantoin', 'colistin'],
}
 
for cls, cols in DRUG_CLASSES.items():
    present = [c for c in cols if c in merged.columns]
    # Count Resistant (==2) per class, ignoring sentinels
    vals = merged[present].replace(SENTINEL, np.nan)
    merged[f'n_resistant_{cls}']    = vals.eq(2).sum(axis=1)
    merged[f'n_tested_{cls}']       = vals.notna().sum(axis=1)
    merged[f'pct_resistant_{cls}']  = (
        merged[f'n_resistant_{cls}'] / merged[f'n_tested_{cls}']
    ).round(3).fillna(0)
 
# Total resistance burden
all_vals = merged[ALL_AB_COLS].replace(SENTINEL, np.nan)
merged['total_resistant']        = all_vals.eq(2).sum(axis=1)
merged['total_antibiotics_tested'] = all_vals.notna().sum(axis=1)
merged['overall_resistance_rate'] = (
    merged['total_resistant'] / merged['total_antibiotics_tested']
).round(3).fillna(0)
 
# Critical MDR flag — resistant to any WHO Reserve antibiotic
merged['is_critical_mdr'] = (
    (merged['imipenem'] == 2) | (merged['colistin'] == 2)
).astype(int)
 
# Cross-resistance flags — clinically meaningful pairs
merged['quinolone_cross_resistance'] = (
    (merged['ciprofloxacin'] == 2) & (merged['nalidixic_acid'] == 2)
).astype(int)
merged['dual_carbapenem_aminoglyco'] = (
    (merged['imipenem'] == 2) &
    ((merged['gentamicin'] == 2) | (merged['amikacin'] == 2))
).astype(int)
 

### Dataset enrichment through CARD database

In [43]:
# Normalise CARD columns
card_df['drug_class_lower']  = card_df['Drug Class'].str.lower().fillna('')
card_df['mechanism_lower']   = card_df['Resistance Mechanism'].str.lower().fillna('')
card_df['gene_family_lower'] = card_df['AMR Gene Family'].str.lower().fillna('')
 
# ── Feature keyword maps ───────────────────────────────────
DRUG_CLASS_FEATURES = {
    'has_gene_penicillin'    : ['penicillin beta-lactam'],
    'has_gene_cephalosporin' : ['cephalosporin'],
    'has_gene_carbapenem'    : ['carbapenem'],
    'has_gene_aminoglycoside': ['aminoglycoside antibiotic'],
    'has_gene_fluoroquinolone': ['fluoroquinolone antibiotic'],
    'has_gene_phenicol'      : ['phenicol antibiotic'],
    'has_gene_sulfonamide'   : ['diaminopyrimidine antibiotic', 'sulfonamide'],
    'has_gene_nitrofuran'    : ['nitrofuran'],
    'has_gene_polymyxin'     : ['peptide antibiotic', 'polymyxin'],
}
 
MECHANISM_FEATURES = {
    'has_efflux'             : ['antibiotic efflux'],
    'has_inactivation'       : ['antibiotic inactivation'],
    'has_target_alteration'  : ['antibiotic target alteration',
                                'antibiotic target replacement'],
    'has_target_protection'  : ['antibiotic target protection'],
    'has_reduced_permeability': ['reduced permeability'],
}
 
GENE_FAMILY_FEATURES = {
    'has_ctx_m'      : ['ctx-m beta-lactamase'],
    'has_tem'        : ['tem beta-lactamase'],
    'has_shv'        : ['shv beta-lactamase'],
    'has_kpc'        : ['kpc beta-lactamase'],
    'has_ndm'        : ['ndm beta-lactamase'],
    'has_vim'        : ['vim beta-lactamase'],
    'has_imp_gene'   : ['imp beta-lactamase'],
    'has_oxa_48'     : ['oxa-48-like beta-lactamase'],
    'has_mcr'        : ['mcr phosphoethanolamine'],
    'has_qnr'        : ['quinolone resistance protein'],
    'has_aac'        : ["aac(6')", "aac(3)", "aac(2')"],
    'has_rnd_efflux' : ['resistance-nodulation-cell division (rnd)'],
    'has_mfs_efflux' : ['major facilitator superfamily (mfs)'],
}
 
ALL_CARD_FEATURE_COLS = (
    list(DRUG_CLASS_FEATURES.keys()) +
    list(MECHANISM_FEATURES.keys()) +
    list(GENE_FAMILY_FEATURES.keys()) +
    ['card_gene_family_count', 'card_drug_class_count']
)
 
# ── Known gene families per species ───────────────────────
# Based on published literature for North African / global
# clinical Enterobacterales isolates
SPECIES_GENE_FAMILIES = {
    'escherichia coli': [
        'tem beta-lactamase', 'ctx-m beta-lactamase', 'shv beta-lactamase',
        'oxa beta-lactamase', "aac(6')", "aac(3)", "aac(2')",
        'quinolone resistance protein (qnr)',
        'mcr phosphoethanolamine transferase',
        'trimethoprim resistant dihydrofolate reductase dfr',
        'major facilitator superfamily (mfs) antibiotic efflux pump',
        'resistance-nodulation-cell division (rnd) antibiotic efflux pump',
    ],
    'klebsiella pneumoniae': [
        'kpc beta-lactamase', 'ctx-m beta-lactamase', 'shv beta-lactamase',
        'tem beta-lactamase', 'oxa beta-lactamase', 'ndm beta-lactamase',
        'oxa beta-lactamase;oxa-48-like beta-lactamase',
        "aac(6')", "aac(3)",
        'quinolone resistance protein (qnr)',
        'resistance-nodulation-cell division (rnd) antibiotic efflux pump',
    ],
    'pseudomonas aeruginosa': [
        'pdc beta-lactamase', 'vim beta-lactamase', 'imp beta-lactamase',
        'oxa beta-lactamase', "aac(6')", "aac(3)",
        'resistance-nodulation-cell division (rnd) antibiotic efflux pump',
    ],
    'acinetobacter baumannii': [
        'oxa beta-lactamase;oxa-51-like beta-lactamase',
        'oxa beta-lactamase;oxa-23-like beta-lactamase',
        'ndm beta-lactamase', 'imp beta-lactamase',
        "aac(6')", "aac(3)",
        'adc beta-lactamases pending classification for carbapenemase activity',
        'resistance-nodulation-cell division (rnd) antibiotic efflux pump',
    ],
    'proteus mirabilis': [
        'tem beta-lactamase', 'ctx-m beta-lactamase',
        "aac(3)", "aac(6')",
        'quinolone resistance protein (qnr)',
        'major facilitator superfamily (mfs) antibiotic efflux pump',
    ],
    'enterobacter cloacae': [
        'act beta-lactamase', 'kpc beta-lactamase', 'ctx-m beta-lactamase',
        "aac(6')",
        'quinolone resistance protein (qnr)',
        'resistance-nodulation-cell division (rnd) antibiotic efflux pump',
    ],
    'staphylococcus aureus': [
        'tem beta-lactamase',
        'erm 23s ribosomal rna methyltransferase',
        "aac(6')", "aph(3')",
        'major facilitator superfamily (mfs) antibiotic efflux pump',
    ],
    'enterococcus faecalis': [
        'vana', 'vanb', 'vanc', 'vand',
        'erm 23s ribosomal rna methyltransferase',
        "aac(6')",
        'major facilitator superfamily (mfs) antibiotic efflux pump',
    ],
    'enterococcus faecium': [
        'vana', 'vanb',
        'erm 23s ribosomal rna methyltransferase',
        "aac(6')",
        'major facilitator superfamily (mfs) antibiotic efflux pump',
    ],
}
 
# ── Extraction function ────────────────────────────────────
def extract_card_features(species_name):
    """
    Return a dict of binary/count CARD features for a species.
    Unknown or NaN species → all zeros (model uses antibiotic cols alone).
    """
    result = {f: 0 for f in ALL_CARD_FEATURE_COLS}
 
    if pd.isna(species_name):
        return result
 
    gene_families = SPECIES_GENE_FAMILIES.get(species_name, [])
    if not gene_families:
        return result
 
    pattern = '|'.join(re.escape(gf) for gf in gene_families)
    mask    = card_df['gene_family_lower'].str.contains(pattern, na=False)
    rows    = card_df[mask]
 
    if rows.empty:
        return result
 
    all_drug = ' '.join(rows['drug_class_lower'].tolist())
    all_mech = ' '.join(rows['mechanism_lower'].tolist())
    all_gene = ' '.join(rows['gene_family_lower'].tolist())
 
    for feat, kws in DRUG_CLASS_FEATURES.items():
        result[feat] = int(any(kw in all_drug for kw in kws))
 
    for feat, kws in MECHANISM_FEATURES.items():
        result[feat] = int(any(kw in all_mech for kw in kws))
 
    for feat, kws in GENE_FAMILY_FEATURES.items():
        result[feat] = int(any(kw in all_gene for kw in kws))
 
    result['card_gene_family_count'] = rows['gene_family_lower'].nunique()
    result['card_drug_class_count']  = rows['drug_class_lower'].nunique()
 
    return result
 
# ── Build species lookup (compute once, map to all rows) ──
unique_species   = merged['bacteria_strain'].unique()
species_lookup   = {sp: extract_card_features(sp) for sp in unique_species}
 
known   = sum(1 for s in unique_species
              if s in SPECIES_GENE_FAMILIES and pd.notna(s))
unknown = sum(1 for s in unique_species
              if s not in SPECIES_GENE_FAMILIES and pd.notna(s))
missing = sum(1 for s in unique_species if pd.isna(s))
 
print(f"  Unique species found   : {len(unique_species)}")
print(f"  Known in CARD map      : {known}  (will get gene features)")
print(f"  Unknown species        : {unknown} (→ zeros, add to map if needed)")
print(f"  NaN - no species       : {missing} (Mendeley rows + missing Souches)")
 
# ── Apply and concat ──────────────────────────────────────
card_features_df = (
    merged['bacteria_strain']
    .map(species_lookup)
    .apply(pd.Series)
)
card_features_df.columns = ALL_CARD_FEATURE_COLS
 
merged = pd.concat(
    [merged.reset_index(drop=True),
     card_features_df.reset_index(drop=True)],
    axis=1
)
 
# Safety fill
merged[ALL_CARD_FEATURE_COLS] = (
    merged[ALL_CARD_FEATURE_COLS].fillna(0).astype(int)
)

print("\nCARD feature totals (rows where feature = 1):")
print(merged[ALL_CARD_FEATURE_COLS]
      .sum().sort_values(ascending=False).to_string())

  Unique species found   : 18
  Known in CARD map      : 5  (will get gene features)
  Unknown species        : 12 (→ zeros, add to map if needed)
  NaN - no species       : 1 (Mendeley rows + missing Souches)

CARD feature totals (rows where feature = 1):
card_drug_class_count       485450
card_gene_family_count      382257
has_gene_cephalosporin        6845
has_mfs_efflux                6845
has_rnd_efflux                6845
has_aac                       6845
has_oxa_48                    6845
has_imp_gene                  6845
has_vim                       6845
has_ndm                       6845
has_kpc                       6845
has_shv                       6845
has_tem                       6845
has_gene_penicillin           6845
has_reduced_permeability      6845
has_target_alteration         6845
has_inactivation              6845
has_efflux                    6845
has_gene_polymyxin            6845
has_gene_sulfonamide          6845
has_gene_phenicol             6845
has_gene

### Sanity Check

In [44]:
# 1. Antibiotic value integrity
print("\n[1] Antibiotic value integrity (only -1/0/1/2 allowed):")
for col in ALL_AB_COLS:
    bad = set(merged[col].unique()) - {-1, 0, 1, 2}
    status = "OK" if not bad else f"BAD: {bad}"
    print(f"    {col:<35} {status}")
 
# 2. Source distribution
print("\n[2] Source distribution:")
print(merged['source'].value_counts().to_string())
 
# 3. Class balance per antibiotic
print("\n[3] R/I/S balance (key antibiotics):")
label_map = {-1:'not_tested', 0:'S', 1:'I', 2:'R'}
for col in ['imipenem', 'ciprofloxacin', 'cefotaxime_ceftriaxone',
            'gentamicin', 'colistin']:
    if col in merged.columns:
        dist = merged[col].map(label_map).value_counts(normalize=True).round(2)
        print(f"    {col:<35} {dist.to_dict()}")
 
# 4. Critical MDR
print(f"\n[4] Critical MDR isolates : "
      f"{merged['is_critical_mdr'].sum()} "
      f"({merged['is_critical_mdr'].mean()*100:.1f}%)")
 
# 5. Column summary
ab_cols   = ALL_AB_COLS
meta_cols = ['bacteria_strain','age','gender','has_diabetes',
             'has_hypertension','prior_hospitalisation','infection_frequency']
eng_cols  = [c for c in merged.columns
             if c.startswith(('n_resistant','n_tested','pct_resistant',
                              'total_','overall_','is_','quinolone_','dual_'))]
card_cols = ALL_CARD_FEATURE_COLS
 
print(f"\n[5] Column breakdown:")
print(f"    Antibiotic labels   : {len(ab_cols)}")
print(f"    Clinical metadata   : {len([c for c in meta_cols if c in merged.columns])}")
print(f"    Engineered features : {len(eng_cols)}")
print(f"    CARD gene features  : {len(card_cols)}")
print(f"    Total columns       : {merged.shape[1]}")
print(f"    Total rows          : {merged.shape[0]}")
 



[1] Antibiotic value integrity (only -1/0/1/2 allowed):
    amoxicillin_ampicillin              OK
    amoxicillin_clavulanate             OK
    cefazolin                           OK
    cefoxitin                           OK
    cefotaxime_ceftriaxone              OK
    imipenem                            OK
    gentamicin                          OK
    amikacin                            OK
    nalidixic_acid                      OK
    ofloxacin                           OK
    ciprofloxacin                       OK
    chloramphenicol                     OK
    cotrimoxazole                       OK
    nitrofurantoin                      OK
    colistin                            OK

[2] Source distribution:
source
kaggle      9957
mendeley     274

[3] R/I/S balance (key antibiotics):
    imipenem                            {'R': 0.56, 'S': 0.41, 'I': 0.02}
    ciprofloxacin                       {'S': 0.82, 'R': 0.16, 'I': 0.02}
    cefotaxime_ceftriaxone              {'R':

### refine duplicate columns

In [47]:

df = merged.copy()
print(f"Loaded: {df.shape}")


# Drop duplicate CARD columns (.1 and .2)
dupe_cols = [c for c in df.columns if c.endswith('.1') or c.endswith('.2')]
df.drop(columns=dupe_cols, inplace=True)
print(f"After dropping dupes: {df.shape[1]} columns")


# Drop infection_frequency (100% NaN)

df.drop(columns=['infection_frequency'], inplace=True)


# Expand species synonym map

EXTRA_SYNONYMS = {
    'e.coi'                : 'escherichia coli',   # typo (432 rows)
    'e.cli'                : 'escherichia coli',   # typo (396 rows)
    'enterobacteria spp.'  : 'escherichia coli',   # generic → map to E. coli
    'enter.bacteria spp.'  : 'escherichia coli',
    'enterobacteria spp'   : 'escherichia coli',
    'enterobacteria spp..' : 'escherichia coli',
    'klbsiella pneumoniae' : 'klebsiella pneumoniae',  # typo (74 rows)
    'morganella morganii'  : 'morganella morganii',    # keep as-is
    'citrobacter spp.'     : 'citrobacter freundii',
    'serratia marcescens'  : 'serratia marcescens',
}

def reclean_species(val):
    if pd.isna(val): return np.nan
    v = str(val).strip().lower()
    return EXTRA_SYNONYMS.get(v, val)

df['bacteria_strain'] = df['bacteria_strain'].apply(reclean_species)
print("\nSpecies after re-cleaning:")
print(df['bacteria_strain'].value_counts(dropna=False).head(12).to_string())


# Re-run CARD enrichment


card_df = pd.read_csv('data/aro_index.tsv', sep='\t')
card_df['drug_class_lower']  = card_df['Drug Class'].str.lower().fillna('')
card_df['mechanism_lower']   = card_df['Resistance Mechanism'].str.lower().fillna('')
card_df['gene_family_lower'] = card_df['AMR Gene Family'].str.lower().fillna('')

# Tighter mapping — each species gets ONLY its characteristic genes
# (not carbapenemase genes for E. coli which rarely has them naturally)
SPECIES_GENE_FAMILIES_TIGHT = {
    'escherichia coli': [
        'tem beta-lactamase', 'ctx-m beta-lactamase',
        "aac(6')", "aac(3)",
        'quinolone resistance protein (qnr)',
        'mcr phosphoethanolamine transferase',
        'trimethoprim resistant dihydrofolate reductase dfr',
        'major facilitator superfamily (mfs) antibiotic efflux pump',
        'resistance-nodulation-cell division (rnd) antibiotic efflux pump',
    ],
    'klebsiella pneumoniae': [
        'kpc beta-lactamase', 'ctx-m beta-lactamase', 'shv beta-lactamase',
        'oxa beta-lactamase;oxa-48-like beta-lactamase',
        "aac(6')",
        'quinolone resistance protein (qnr)',
        'resistance-nodulation-cell division (rnd) antibiotic efflux pump',
    ],
    'pseudomonas aeruginosa': [
        'pdc beta-lactamase', 'vim beta-lactamase', 'imp beta-lactamase',
        "aac(6')",
        'resistance-nodulation-cell division (rnd) antibiotic efflux pump',
    ],
    'acinetobacter baumannii': [
        'oxa beta-lactamase;oxa-51-like beta-lactamase',
        'oxa beta-lactamase;oxa-23-like beta-lactamase',
        'ndm beta-lactamase',
        'adc beta-lactamases pending classification for carbapenemase activity',
        'resistance-nodulation-cell division (rnd) antibiotic efflux pump',
    ],
    'proteus mirabilis': [
        'tem beta-lactamase',
        "aac(3)",
        'quinolone resistance protein (qnr)',
        'major facilitator superfamily (mfs) antibiotic efflux pump',
    ],
    'morganella morganii': [
        'tem beta-lactamase',
        "aac(3)", "aac(6')",
        'major facilitator superfamily (mfs) antibiotic efflux pump',
    ],
    'citrobacter freundii': [
        'ctx-m beta-lactamase', 'act beta-lactamase',
        "aac(6')",
        'resistance-nodulation-cell division (rnd) antibiotic efflux pump',
    ],
    'serratia marcescens': [
        'ctx-m beta-lactamase',
        "aac(6')",
        'resistance-nodulation-cell division (rnd) antibiotic efflux pump',
    ],
    'enterobacter cloacae': [
        'act beta-lactamase', 'kpc beta-lactamase',
        "aac(6')",
        'resistance-nodulation-cell division (rnd) antibiotic efflux pump',
    ],
}

DRUG_CLASS_FEATURES = {
    'has_gene_penicillin'    : ['penicillin beta-lactam'],
    'has_gene_cephalosporin' : ['cephalosporin'],
    'has_gene_carbapenem'    : ['carbapenem'],
    'has_gene_aminoglycoside': ['aminoglycoside antibiotic'],
    'has_gene_fluoroquinolone': ['fluoroquinolone antibiotic'],
    'has_gene_phenicol'      : ['phenicol antibiotic'],
    'has_gene_sulfonamide'   : ['diaminopyrimidine antibiotic'],
    'has_gene_polymyxin'     : ['peptide antibiotic'],
}
MECHANISM_FEATURES = {
    'has_efflux'             : ['antibiotic efflux'],
    'has_inactivation'       : ['antibiotic inactivation'],
    'has_target_alteration'  : ['antibiotic target alteration',
                                'antibiotic target replacement'],
    'has_target_protection'  : ['antibiotic target protection'],
    'has_reduced_permeability': ['reduced permeability'],
}
GENE_FAMILY_FEATURES = {
    'has_ctx_m'     : ['ctx-m beta-lactamase'],
    'has_tem'       : ['tem beta-lactamase'],
    'has_shv'       : ['shv beta-lactamase'],
    'has_kpc'       : ['kpc beta-lactamase'],
    'has_ndm'       : ['ndm beta-lactamase'],
    'has_vim'       : ['vim beta-lactamase'],
    'has_imp_gene'  : ['imp beta-lactamase'],
    'has_oxa_48'    : ['oxa-48-like beta-lactamase'],
    'has_mcr'       : ['mcr phosphoethanolamine'],
    'has_qnr'       : ['quinolone resistance protein'],
    'has_aac'       : ["aac(6')", "aac(3)"],
    'has_rnd_efflux': ['resistance-nodulation-cell division (rnd)'],
    'has_mfs_efflux': ['major facilitator superfamily (mfs)'],
}

ALL_CARD_COLS = (list(DRUG_CLASS_FEATURES) + list(MECHANISM_FEATURES) +
                 list(GENE_FAMILY_FEATURES) +
                 ['card_gene_family_count', 'card_drug_class_count'])

# Drop old CARD columns first
old_card = [c for c in df.columns if c in ALL_CARD_COLS]
df.drop(columns=old_card, inplace=True)

def extract_card_features(species):
    result = {f: 0 for f in ALL_CARD_COLS}
    if pd.isna(species): return result
    gene_families = SPECIES_GENE_FAMILIES_TIGHT.get(species, [])
    if not gene_families: return result
    pattern = '|'.join(re.escape(g) for g in gene_families)
    rows = card_df[card_df['gene_family_lower'].str.contains(pattern, na=False)]
    if rows.empty: return result
    all_drug = ' '.join(rows['drug_class_lower'])
    all_mech = ' '.join(rows['mechanism_lower'])
    all_gene = ' '.join(rows['gene_family_lower'])
    for feat, kws in DRUG_CLASS_FEATURES.items():
        result[feat] = int(any(k in all_drug for k in kws))
    for feat, kws in MECHANISM_FEATURES.items():
        result[feat] = int(any(k in all_mech for k in kws))
    for feat, kws in GENE_FAMILY_FEATURES.items():
        result[feat] = int(any(k in all_gene for k in kws))
    result['card_gene_family_count'] = rows['gene_family_lower'].nunique()
    result['card_drug_class_count']  = rows['drug_class_lower'].nunique()
    return result

lookup = {sp: extract_card_features(sp) for sp in df['bacteria_strain'].unique()}
card_features = df['bacteria_strain'].map(lookup).apply(pd.Series)
card_features.columns = ALL_CARD_COLS
df = pd.concat([df.reset_index(drop=True),
                card_features.reset_index(drop=True)], axis=1)
df[ALL_CARD_COLS] = df[ALL_CARD_COLS].fillna(0).astype(int)


# FIX 5 — Drop columns useless for modelling

DROP_COLS = [
    'patient_id', 'patient_name', 'patient_email',
    'patient_address', 'age_gender',    # PII + already parsed
    'collection_date',                  # 783 NaN, not useful as-is
    'sample_location',                  # 97% NaN
    'notes',                            # free text
    'bacteria_strain_raw',              # replaced by bacteria_strain
]
df.drop(columns=[c for c in DROP_COLS if c in df.columns], inplace=True)


# FINAL REPORT

print(f"\n=== FINAL SHAPE: {df.shape} ===")
print(f"\nColumn groups:")
ab   = ['amoxicillin_ampicillin','amoxicillin_clavulanate','cefazolin','cefoxitin',
        'cefotaxime_ceftriaxone','imipenem','gentamicin','amikacin','nalidixic_acid',
        'ofloxacin','ciprofloxacin','chloramphenicol','cotrimoxazole',
        'nitrofurantoin','colistin']
meta = ['bacteria_strain','age','gender','has_diabetes',
        'has_hypertension','prior_hospitalisation']
eng  = [c for c in df.columns if any(c.startswith(p) for p in
        ['n_resistant','n_tested','pct_','total_','overall_','is_',
         'quinolone_cross','dual_'])]
card = ALL_CARD_COLS

print(f"  Antibiotic labels   : {len([c for c in ab if c in df.columns])}")
print(f"  Clinical metadata   : {len([c for c in meta if c in df.columns])}")
print(f"  Engineered features : {len([c for c in eng if c in df.columns])}")
print(f"  CARD gene features  : {len([c for c in card if c in df.columns])}")
print(f"  Total               : {df.shape[1]}")

print(f"\nRemaining NaNs:")
miss = df.isnull().sum()
print(miss[miss > 0].to_string())

print(f"\nCARD feature variance check:")
for col in list(GENE_FAMILY_FEATURES.keys()):
    vc = df[col].value_counts(normalize=True).round(3).to_dict()
    print(f"  {col:<20} {vc}")



Loaded: (10231, 73)
After dropping dupes: 73 columns

Species after re-cleaning:
bacteria_strain
escherichia coli           6985
proteus mirabilis           644
klebsiella pneumoniae       639
citrobacter freundii        481
morganella morganii         305
NaN                         284
serratia marcescens         256
pseudomonas aeruginosa      200
acinetobacter baumannii     181
enteobacteria spp.           95
klebsie.lla pneumoniae       63
protus mirabilis             51

=== FINAL SHAPE: (10231, 69) ===

Column groups:
  Antibiotic labels   : 15
  Clinical metadata   : 6
  Engineered features : 18
  CARD gene features  : 28
  Total               : 69

Remaining NaNs:
bacteria_strain          284
age                      284
gender                   284
has_diabetes             284
has_hypertension         284
prior_hospitalisation    284

CARD feature variance check:
  has_ctx_m            {1: 0.947, 0: 0.053}
  has_tem              {1: 0.93, 0: 0.07}
  has_shv              {1: 0

### Save the updated Dataset

In [48]:
df.to_csv('data/processed/amr_final.csv', index=False)
print(f"\nSaved → data/processed/amr_final.csv")


Saved → data/processed/amr_final.csv
